# 02 — Convex regime: logistic regression & linear SVM

This notebook is the **core of the course-relevant analysis**. The loss is strictly convex, so we can talk meaningfully about:

- step-size choices and their relation to the smoothness constant $L$,
- convergence rates: GD is $\mathcal{O}(1/k)$, Nesterov $\mathcal{O}(1/k^2)$, L-BFGS super-linear locally,
- the role of the $\ell_2$ regularization in making the problem $\mu$-strongly convex (linear rate),
- how SGD trades a slower (sub-linear) rate for cheap per-iteration cost.

We run **the same model** with several optimizers and compare loss vs. number of optimizer steps.

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))
import numpy as np
import matplotlib.pyplot as plt
from src.data_loader import load_cifar10_numpy, NUM_CLASSES
from src.convex_models import (
    logistic_loss_and_grad, svm_loss_and_grad, train_linear_model,
)

In [ ]:
X_train, y_train, X_test, y_test = load_cifar10_numpy(
    flatten=True, normalize=True, subset=10_000, seed=0,
)
X_train.shape, X_test.shape

## 2.1 Logistic regression — optimizer race

In [ ]:
configs = [
    ('sgd',      {'lr': 1e-1}),
    ('momentum', {'lr': 1e-1, 'momentum': 0.9}),
    ('nesterov', {'lr': 1e-1, 'momentum': 0.9}),
    ('adagrad',  {'lr': 1e-1}),
    ('rmsprop',  {'lr': 1e-2}),
    ('adam',     {'lr': 1e-2}),
]
histories = {}
for name, kw in configs:
    print(f'--- {name} ---')
    _, h = train_linear_model(
        X_train, y_train, X_test, y_test,
        loss_fn=logistic_loss_and_grad, num_classes=NUM_CLASSES,
        optimizer=name, optim_kwargs=kw,
        epochs=15, batch_size=256, l2=1e-4, eval_every=20, verbose=False,
    )
    histories[name] = h

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for name, h in histories.items():
    axes[0].plot(h.step_idx, h.losses, label=name)
    axes[1].plot(range(1, len(h.test_acc) + 1), h.test_acc, label=name)
axes[0].set(xlabel='step', ylabel='loss', yscale='log', title='Logistic regression — loss')
axes[1].set(xlabel='epoch', ylabel='test accuracy', title='Logistic regression — test acc')
for ax in axes:
    ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()

## 2.2 Linear SVM (squared hinge) — same optimizer race

In [ ]:
svm_histories = {}
for name, kw in configs:
    print(f'--- {name} ---')
    _, h = train_linear_model(
        X_train, y_train, X_test, y_test,
        loss_fn=svm_loss_and_grad, num_classes=NUM_CLASSES,
        optimizer=name, optim_kwargs=kw,
        epochs=15, batch_size=256, l2=1e-4, eval_every=20, verbose=False,
    )
    svm_histories[name] = h

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for name, h in svm_histories.items():
    axes[0].plot(h.step_idx, h.losses, label=name)
    axes[1].plot(range(1, len(h.test_acc) + 1), h.test_acc, label=name)
axes[0].set(xlabel='step', ylabel='loss', yscale='log', title='Linear SVM — loss')
axes[1].set(xlabel='epoch', ylabel='test accuracy', title='Linear SVM — test acc')
for ax in axes:
    ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()

## 2.3 Second-order baseline — L-BFGS on full batch

L-BFGS uses a low-memory Hessian approximation and converges super-linearly locally on strictly convex problems. We use scipy's implementation as a reference point: it should reach a much lower loss in far fewer iterations than any of the first-order methods above.

In [ ]:
from scipy.optimize import minimize
from src.convex_models import _augment_bias, _one_hot, logistic_loss_and_grad

X_aug = _augment_bias(X_train)
Y_oh = _one_hot(y_train, NUM_CLASSES)
D, K = X_aug.shape[1], NUM_CLASSES

def obj(w_flat):
    W = w_flat.reshape(D, K)
    loss, grad = logistic_loss_and_grad(W, X_aug, Y_oh, l2=1e-4)
    return loss, grad.ravel()

w0 = np.zeros(D * K, dtype=np.float64)
result = minimize(obj, w0, jac=True, method='L-BFGS-B', options={'maxiter': 200, 'disp': True})
print('final loss:', result.fun)

## 2.4 Notes for the report

- Add a table: optimizer × {final loss, final test acc, # steps to reach loss = X}.
- Plot loss in **log scale** to expose the asymptotic rate (slope = rate).
- Discuss the link between the empirical curves and the theoretical bounds: e.g. why Nesterov is visibly faster than plain momentum on a strongly convex problem, and why Adam looks competitive even though it has *no* theoretical advantage on convex problems.
- Compare to L-BFGS to motivate why 2nd-order methods are the gold standard in the convex setting — and why they fail to scale to the CNN setting (next notebook).